# 05 — Decision Tree Regression
## Predicting house prices with a Decision Tree

**Objective:** Fit a Decision Tree Regressor, evaluate training and test performance, visualize the model, and investigate how tree depth affects overfitting.

**Dataset:** `Practical_Dataset for ML.xlsx`, sheet `Data`  
**Target:** `price_etb`  
**Features:** numeric predictors, excluding target and identifier `house_id`  
**Split:** 80% training / 20% test, `random_state=42`

A Decision Tree learns a sequence of feature-based splits. It can model nonlinear relationships and interactions, but a deep tree may memorize training data. We will compare a default tree with a tree selected using training-only cross-validation.

## 1. Import libraries and load the workbook

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "price_etb"
DATA_FILENAME = "Practical_Dataset for ML.xlsx"

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
candidates = [
    PROJECT_ROOT / "data" / DATA_FILENAME,
    PROJECT_ROOT / DATA_FILENAME,
    cwd / "data" / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Put it in ML_Course/data/ "
        "or directly inside ML_Course, then rerun this cell."
    )

df = pd.read_excel(DATA_PATH, sheet_name="Data")
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
def evaluate_train_test(model, X_train, y_train, X_test, y_test):
    """Return standard regression metrics for training and test data."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    def calculate_metrics(y_true, y_pred):
        mse = mean_squared_error(y_true, y_pred)
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y_true, y_pred),
        }

    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    return pd.DataFrame([
        {"Dataset": "Train", **calculate_metrics(y_train, train_predictions)},
        {"Dataset": "Test", **calculate_metrics(y_test, test_predictions)},
    ]).set_index("Dataset")


def interpret_metrics(metrics):
    """Print a short interpretation of train/test regression metrics."""
    if not {"Train", "Test"}.issubset(set(metrics.index)):
        print("Metrics table does not contain both Train and Test rows.")
        return

    train = metrics.loc["Train"]
    test = metrics.loc["Test"]

    print(f"Train RMSE: {train['RMSE']:,.2f} ETB")
    print(f"Test RMSE:  {test['RMSE']:,.2f} ETB")
    print(f"Train R²:   {train['R2']:.4f}")
    print(f"Test R²:    {test['R2']:.4f}")

    if test["R2"] < 0:
        print("Interpretation: test R² is negative; the model performs worse than predicting the test-set mean.")
    elif train["R2"] - test["R2"] > 0.15:
        print("Interpretation: the train/test R² gap is noticeable; investigate possible overfitting.")
    else:
        print("Interpretation: compare train/test errors and R² to judge generalization.")


## 2. Select the target and numeric features

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Target column {TARGET!r} not found.")

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
FEATURES = [c for c in numeric_columns if c not in {TARGET, "house_id"}]

if not FEATURES:
    raise ValueError("No numeric predictor columns available.")

print("Target:", TARGET)
print("Features:", FEATURES)
print("Missing values in features:")
display(df[FEATURES].isna().sum().sort_values(ascending=False))
print("Missing target values:", df[TARGET].isna().sum())


## 3. Create the train/test split

Split the data before fitting the imputer or model. Reuse the same split settings in notebooks 01–09 for fair comparison. If your earlier notebooks used different split settings or filtered rows differently, align them before making a combined model comparison.

In [ ]:
model_df = df[FEATURES + [TARGET]].dropna(subset=[TARGET]).copy()
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Total rows:", len(model_df))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 4. Fit and evaluate a baseline Decision Tree

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor

baseline_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE))
])

baseline_tree.fit(X_train, y_train)
baseline_metrics = evaluate_train_test(
    baseline_tree, X_train, y_train, X_test, y_test
)

display(baseline_metrics.round(4))
interpret_metrics(baseline_metrics)


### Interpretation

- **MAE / RMSE:** lower values mean smaller prediction errors, measured in ETB.
- **R²:** higher is generally better; a negative test R² means the model is worse than predicting the test-set mean.
- If training error is extremely low but test error is much higher, the unrestricted tree may be overfitting.
- Decision trees do not require feature scaling, but missing-value handling is still important.

## 5. Tune tree complexity with cross-validation

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE))
])

param_grid = {
    "tree__max_depth": [3, 5, 8, 12, None],
    "tree__min_samples_split": [2, 10, 30],
    "tree__min_samples_leaf": [1, 5, 15, 30],
}

tree_search = GridSearchCV(
    tree_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)
tree_search.fit(X_train, y_train)

print("Best parameters:", tree_search.best_params_)
print(f"Best mean CV MAE: {-tree_search.best_score_:,.2f} ETB")
tuned_tree = tree_search.best_estimator_


## 6. Evaluate the tuned Decision Tree

In [ ]:
tuned_metrics = evaluate_train_test(
    tuned_tree, X_train, y_train, X_test, y_test
)
display(tuned_metrics.round(4))
interpret_metrics(tuned_metrics)

tree_comparison = pd.concat([
    baseline_metrics.assign(Model="Baseline Decision Tree").reset_index(),
    tuned_metrics.assign(Model="Tuned Decision Tree").reset_index()
], ignore_index=True)

display(
    tree_comparison[["Model", "Dataset", "MAE", "MSE", "RMSE", "R2"]]
    .sort_values(["Model", "Dataset"])
    .round(3)
)


## 7. Visualize the top of the fitted tree

In [ ]:
from sklearn.tree import plot_tree

fitted_tree = tuned_tree.named_steps["tree"]
imputer = tuned_tree.named_steps["imputer"]
feature_names = FEATURES

plt.figure(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    filled=False,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title("Tuned Decision Tree (display limited to first 3 levels)")
plt.tight_layout()
plt.show()


## 8. Feature importance

In [ ]:
importance = pd.Series(
    fitted_tree.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

display(importance.to_frame("Importance"))

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Decision Tree feature importance")
plt.title("Feature Importance — Tuned Decision Tree")
plt.tight_layout()
plt.show()


**Interpretation caution:** tree feature importance indicates how much a feature contributes to splits in this fitted tree. It is not proof of causation, and importance can be biased or unstable when predictors are correlated. Validate important features with domain knowledge and, if appropriate, permutation importance.

## 9. Residual and prediction diagnostics

In [ ]:
test_predictions = tuned_tree.predict(X_test)
residuals = y_test.to_numpy() - test_predictions

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(test_predictions, residuals, alpha=0.55)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted price (ETB)")
axes[0].set_ylabel("Residual: actual - predicted (ETB)")
axes[0].set_title("Decision Tree residuals")

axes[1].scatter(y_test, test_predictions, alpha=0.55)
low = min(float(y_test.min()), float(test_predictions.min()))
high = max(float(y_test.max()), float(test_predictions.max()))
axes[1].plot([low, high], [low, high], linestyle="--")
axes[1].set_xlabel("Actual price (ETB)")
axes[1].set_ylabel("Predicted price (ETB)")
axes[1].set_title("Actual vs predicted test prices")
plt.tight_layout()
plt.show()


## 10. Save metrics and tuning results

In [ ]:
metrics_dir = PROJECT_ROOT / "outputs" / "metrics"
plots_dir = PROJECT_ROOT / "outputs" / "plots"
metrics_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

baseline_metrics.to_csv(metrics_dir / "05_decision_tree_baseline_metrics.csv")
tuned_metrics.to_csv(metrics_dir / "05_decision_tree_tuned_metrics.csv")
tree_comparison.to_csv(metrics_dir / "05_decision_tree_comparison.csv", index=False)
pd.DataFrame(tree_search.cv_results_).to_csv(
    metrics_dir / "05_decision_tree_grid_search_results.csv", index=False
)
importance.rename("Importance").to_csv(metrics_dir / "05_decision_tree_feature_importance.csv")
print("Saved Decision Tree results to:", metrics_dir)


## 11. Conclusion checklist

After running the notebook, write a conclusion using your actual results:
1. What were the baseline and tuned test MAE, RMSE, and R²?
2. Which hyperparameters were selected by cross-validation?
3. Did tuning reduce the train/test performance gap?
4. Which features had the highest tree importance?
5. How does this model compare with SLR, MLR, Polynomial Regression, and SVR on the same test rows?
6. What limitations remain?

Do not invent metric values. A tuned Decision Tree is not automatically better than other models.